In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/archive/diabetes.csv')
df.dropna().drop_duplicates

df.head()

In [ ]:
zeroes_as_missing_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'Age']
df[zeroes_as_missing_cols] = df[zeroes_as_missing_cols].replace(0, np.nan)

df.head()

In [ ]:
df = df.dropna(subset='Glucose')

df['BloodPressure'] = df['BloodPressure'].fillna(df['BloodPressure'].median())

df['SkinThickness'] = df['SkinThickness'].fillna(df['SkinThickness'].median())

df['Insulin'] = df['Insulin'].fillna(df['Insulin'].median())

df = df.dropna(subset='BMI')

df = df.reset_index(drop=True)
df['Patient_ID'] = df.index.astype(str)

print(df['Outcome'].unique())

df.head()

In [ ]:
df[df["Patient_ID"] == "0"]

In [ ]:
# creating the functions that can query the dataframe

from agents import Agent, function_tool

@function_tool
def get_patient_record(patient_id: str) -> dict:
    patient_record = df[df['Patient_ID'] == patient_id]

    if patient_record.empty:
        return {"error": "patient not found"}
    
    patient_record = patient_record.iloc[0].to_dict()

    return(patient_record)

@function_tool
def get_cohort_average(age_low: int, age_high: int) -> dict:
    cohort = df[(df['Age'] >= age_low) & (df['Age'] <= age_high)]

    if cohort.empty:
        return {"error": "no patients in this age range"}
    
    cohort_average = cohort[['Glucose', 'BMI', 'BloodPressure']].mean().to_dict()

    return cohort_average


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print(os.getenv("OPENAI_API_KEY"))

In [ ]:
from pydantic import BaseModel, Field 
from enum import Enum

class RiskLevel(str, Enum):
    low = "Low"
    medium = "Medium"
    high = "High"

class RiskAssessment(BaseModel):
    patient_id : str
    risk_level : RiskLevel
    key_factors : list[str] = Field(description="Which attributes are driving the risk level")
    reasoning : str
    used_cohort_context : bool = Field(description="Whether cohort comparison was needed")


In [ ]:
from openai import OpenAI
from agents import Agent, Runner

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

risk_agent = Agent(
    name="Risk",
    instructions="You are a helpful patient diabetes risk analysis agent. Call get_patient_record first. Only call get_cohort_average if a value looks borderline.",
    model="gpt-4o-mini",
    tools=[get_patient_record, get_cohort_average],
    output_type=RiskAssessment
)

In [ ]:
class Recommendation(BaseModel):
    patient_id: str
    priority_actions: list[str] = Field(description="2-4 concrete, prioritized actions")
    rationale: str

In [ ]:
prompt = """You are a lifestyle recommendation agent for diabetes risk management.

You will receive a structured risk assessment containing a risk level, a list of
key contributing factors, and reasoning. Your job is to produce 2-4 concrete,
prioritized actions that specifically target the listed key factors — not
generic health advice.

Rules:
- Every recommendation must connect directly to one of the key_factors listed.
  If BMI is listed as a factor, address BMI specifically. If glucose is listed,
  address glucose specifically. Don't give advice about factors that weren't flagged.
- Be concrete and actionable, not vague. "Improve your diet" is not acceptable.
  "Reduce refined carbohydrate intake, which directly affects blood glucose spikes"
  is acceptable.
- Prioritize the actions — list the most impactful one first, based on which
  factor seems most significant given the risk level and reasoning provided.
- Do not provide medical diagnoses, medication advice, or anything requiring a
  doctor's judgment. Stay in the lane of lifestyle/behavioral guidance.
- Briefly explain WHY each action matters given this specific patient's profile,
  not just what to do.
"""

recommendation_agent = Agent(
    name="Recommendation",
    instructions=prompt,
    model="gpt-4o-mini",
    output_type=Recommendation
)

In [ ]:
class Intent(str, Enum):
    risk_only = "risk_only"
    recommendation_only = "recommendation_only"
    recommendation_needs_risk_first = "recommendation_needs_risk_first"

class OrchestratorDecision(BaseModel):
    intent: Intent
    patient_id: str
    reasoning: str

In [ ]:
orchestration_prompt = """Classify a diabetes-related query into one of:
    - risk_only: user wants to know how risky their profile looks
    - recommendation_only: user wants advice and a risk assessment already exists
    - recommendation_needs_risk_first: user wants advice but no assessment exists yet
    Extract the patient_id mentioned in the query."""

orchestrator_agent = Agent(
    name="Orchestrator",
    instructions=orchestration_prompt,
    model="gpt-4o-mini",
    output_type=OrchestratorDecision
)

In [ ]:
# orchestration logic

async def orchestrate(user_query: str):
    decision_result = await Runner.run(orchestrator_agent, user_query)
    decision = decision_result.final_output

    if decision.intent == Intent.risk_only:
        risk_result = await Runner.run(risk_agent, f"Assess risk for patient_id={decision.patient_id}")
        return risk_result.final_output

    risk_result = await Runner.run(risk_agent, f"Assess risk for patient_id={decision.patient_id}")
    assessment = risk_result.final_output

    recommendation_result = await Runner.run(recommendation_agent, assessment.model_dump_json())
    return recommendation_result.final_output

In [ ]:
# Test

def print_risk_assessment(a):
    if isinstance(result, RiskAssessment):
        print(f"Patient: {result.patient_id}")
        print(f"Risk level: {result.risk_level.value}")
        print("Key factors:")
        for factor in result.key_factors:
            print(f"  - {factor}")
        print(f"Reasoning: {result.reasoning}")

    elif isinstance(result, Recommendation):
        print(f"Patient: {result.patient_id}")
        print("Priority actions:")
        for i, action in enumerate(result.priority_actions, 1):
            print(f"  {i}. {action}")
        print(f"Rationale: {result.rationale}")

    else:
        print(result)

result = await orchestrate("What is patient 0s risk level?")

print_risk_assessment(result)